# 🎙️ Fine-Tune Whisper Tiếng Việt Bằng Giọng Nói Của Bạn (Google Colab T4)

Notebook này sử dụng tập dữ liệu ghi âm **dataset.zip** mà bạn vừa thu âm để fine-tune mô hình **OpenAI Whisper (Small)** bằng kỹ thuật **LoRA (PEFT)**.

### Kết quả đạt được:
- Whisper học trực tiếp chất giọng, ngữ điệu và phát âm của bạn.
- Đã khắc phục triệt để lỗi PyTorch backward `element 0 does not require grad`.
- Tự động nạp đè vào `CTranslate2 (int8_float16)` (~450MB) để copy về laptop Arch Linux chạy ngoại tuyến 100%.

## 1. Kiểm tra GPU & Cài đặt thư viện ổn định

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q "transformers>=4.38.0,<4.43.0" datasets evaluate jiwer soundfile librosa accelerate peft bitsandbytes "ctranslate2>=4.0.0"


## 2. Giải nén dataset.zip
> **Lưu ý**: Hãy kéo thả file `dataset.zip` từ máy tính vào thanh Files bên trái của Colab trước khi chạy ô này!

In [ ]:
import os
import json

if os.path.exists("dataset.zip"):
    !unzip -q -o dataset.zip
    print("✅ Đã giải nén dataset.zip thành công!")
else:
    print("⚠️ Chưa tìm thấy file dataset.zip. Hãy upload file lên Colab trước!")

# Kiểm tra số lượng mẫu thu âm
dataset_dir = "dataset"
metadata_file = os.path.join(dataset_dir, "metadata.jsonl")
samples = []
if os.path.exists(metadata_file):
    with open(metadata_file, "r", encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"🎯 Đã tìm thấy {len(samples)} mẫu câu lệnh với giọng nói của bạn!")
    print("Mẫu đầu tiên:", samples[0])
else:
    print("❌ Không tìm thấy file metadata.jsonl trong thư mục dataset/")

## 3. Chuẩn bị Hugging Face Dataset

In [ ]:
import soundfile as sf
import librosa
import torch
from datasets import Dataset, Audio
from transformers import WhisperProcessor

MODEL_ID = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Vietnamese", task="transcribe")

data_records = []
for s in samples:
    audio_path = os.path.join(dataset_dir, s["audio"])
    if os.path.exists(audio_path):
        data_records.append({"audio": audio_path, "sentence": s["sentence"]})

raw_dataset = Dataset.from_list(data_records)
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

prepared_dataset = raw_dataset.map(prepare_dataset, remove_columns=raw_dataset.column_names, num_proc=1)
print("✅ Chuẩn bị xong Dataset:", prepared_dataset)


## 4. Cấu hình LoRA (PEFT) & Huấn luyện (Fine-Tuning)
> **Đã fix**: Tích hợp `enable_input_require_grads()` và learning rate cân bằng `3e-5` chống Catastrophic Forgetting.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

start_token_id = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=start_token_id)

# Nạp mô hình Whisper Small
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, device_map="auto")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False
model.enable_input_require_grads()

# Áp dụng LoRA Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    num_train_epochs=8,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=5,
    save_strategy="no",
    report_to=["none"]
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=prepared_dataset,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print("🚀 Bắt đầu huấn luyện...")
trainer.train()
print("🎉 Huấn luyện thành công!")


## 5. Hợp nhất LoRA & Xuất sang CTranslate2 (int8_float16)

In [ ]:
# Hợp nhất adapter LoRA vào mô hình gốc
merged_dir = "./whisper-small-merged"
merged_model = model.merge_and_unload()
merged_model.save_pretrained(merged_dir)
processor.save_pretrained(merged_dir)

# Chuyển đổi sang CTranslate2 int8_float16
ct2_dir = "./whisper-small-vietnamese-ct2"
!ct2-transformers-converter --model {merged_dir} --output_dir {ct2_dir} --quantization int8_float16 --force
!zip -r whisper-small-vietnamese-ct2.zip {ct2_dir}

print("=" * 60)
print("✅ HOÀN THÀNH 100%!")
print("👉 Hãy tải file 'whisper-small-vietnamese-ct2.zip' từ thanh Files về laptop của bạn!")
print("=" * 60)